# Training data

In [1]:
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'

In [32]:
""" 
IMPORTS
"""
import os
import numpy as np
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --Machine learning and statistics

import pickle
from one.api import ONE
import concurrent.futures

# Get my functions
from functions import create_grouped_gradient_palette
# Get my functions
functions_path = prefix + '/representation_learning_variability/Functions/'
os.chdir(functions_path)
from one_functions_generic import download_subjectTables

one = ONE(mode='remote')

In [79]:
# Get my functions
def idxs_from_files(design_matrices):
    
    idxs = []
    mouse_names = []
    for m, mat in enumerate(design_matrices):
        mouse_name = design_matrices[m][51:]
        eid = design_matrices[m][14:50]
        idx = str(eid + '_' + mouse_name)

        if len(idxs) == 0:
            idxs = idx
            mouse_names = mouse_name
        else:
            idxs = np.hstack((idxs, idx))
            mouse_names = np.hstack((mouse_names, mouse_name))
            
    return idxs, mouse_names


""" 
LOAD DATA AND PARAMETERS
"""
# LOAD DATA

data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/'
data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices'

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

learning_data_path =   prefix + 'representation_learning_variability/paper-individuality/data/training_data/'
learning_data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/training_data/'


In [80]:
all_files = os.listdir(learning_data_path)
mice_to_download = []

for m, mouse_name in enumerate(np.unique(mouse_names)):
    
    filename = 'training_data_trials_'+mouse_name
    if filename not in all_files:
        mice_to_download.append(mouse_name)
    
print(len(mice_to_download))

102


# Download and save data

## When there are multiple versions I take the most recent... not sure how correct this is

In [102]:
import re
from pathlib import Path
from datetime import datetime

def get_latest_ibl_paths(path_list):
    """
    Filters a list of PosixPaths to return only the most recent patched versions.
    """
    latest_files = {}

    for path in path_list:
        path_str = str(path)
        
        # 1. Extract date from #YYYY-MM-DD# using regex
        match = re.search(r'#(\d{4}-\d{2}-\d{2})#', path_str)
        current_date = datetime.strptime(match.group(1), '%Y-%m-%d') if match else datetime.min
        
        # 2. Identify the unique dataset key (filename)
        # We strip out the UUID and the patch folder to identify "matching" datasets
        # e.g., 'CSHL052/_ibl_subjectTrials.table'
        dataset_key = path.name.split('.')[0] + "_" + path.parent.parent.name if match else path.name.split('.')[0] + "_" + path.parent.name
        
        # 3. Keep the one with the most recent date
        if dataset_key not in latest_files:
            latest_files[dataset_key] = (current_date, path)
        else:
            if current_date > latest_files[dataset_key][0]:
                latest_files[dataset_key] = (current_date, path)

    return [val[1] for val in latest_files.values()]



In [118]:
for m, mouse_name in enumerate(mice_to_download):
    # Trials data
    print(mouse_name)
    files = download_subjectTables(one, mouse_name, trials=True, training=True,
                        target_path=None, tag=None, overwrite=False, check_updates=True)
    trial_files, training_files = files
    trial_file = get_latest_ibl_paths(trial_files)
    training_file = get_latest_ibl_paths(training_files)

    trials = pd.read_parquet(trial_file)
    training = pd.read_parquet(training_file)
    training = training.reset_index()
    trained_date = pd.to_datetime(training.loc[training['training_status'].isin(['trained 1a', 'trained 1b']), 'date']).dt.date
    
    sessions = trials[['task_protocol', 'session_start_time', 'session']].dropna().drop_duplicates()
    training_sessions = sessions.loc[sessions['task_protocol'].str.contains('training')]
    trials['session_date'] = pd.to_datetime(trials['session_start_time']).dt.date
    mouse_data = trials.loc[trials['session_date']<np.min(trained_date)]

    # Save data
    mouse_data.to_parquet(learning_data_path+'training_data_trials_'+mouse_name,compression='gzip') 
    print('Successful for mouse ' + mouse_name)
    print(len(mouse_data))


CSHL045


File aggregates/Subjects/churchlandlab/CSHL045/#2025-03-03#/_ibl_subjectTrials.table.dd602a2e-97dd-4c42-ac57-c14a57c6c87d.pqt not found on ibl-brain-wide-map-private


Successful for mouse CSHL045
30599
CSHL047


File aggregates/Subjects/churchlandlab/CSHL047/#2025-03-03#/_ibl_subjectTrials.table.2edc7114-e8a2-4ad6-afc6-ad5d0f13e340.pqt not found on ibl-brain-wide-map-private


Successful for mouse CSHL047
18477
CSHL049


...

Successful for mouse DY_016
3417
DY_018


File aggregates/Subjects/danlab/DY_018/#2025-03-03#/_ibl_subjectTrials.table.92fd4480-b1c5-4e9b-999b-46c15de18034.pqt not found on ibl-brain-wide-map-private


Successful for mouse DY_018
8856
DY_020


File aggregates/Subjects/danlab/DY_020/#2025-03-03#/_ibl_subjectTrials.table.c310e989-8063-4dc2-9612-9cb50040172f.pqt not found on ibl-brain-wide-map-private


Successful for mouse DY_020
12957
NR_0017
Successful for mouse NR_0017
15909
NR_0019
Successful for mouse NR_0019
17197
NR_0020
Successful for mouse NR_0020
9565
NR_0021
Successful for mouse NR_0021
12504
NR_0027
Successful for mouse NR_0027
17867
NR_0028
Successful for mouse NR_0028
7282
NYU-11
Successful for mouse NYU-11
11668
NYU-12
Successful for mouse NYU-12
14582
NYU-21
Successful for mouse NYU-21
3473
NYU-30
Successful for mouse NYU-30
4737
NYU-39
Successful for mouse NYU-39
4721
NYU-40
Successful for mouse NYU-40
6136
NYU-45
Successful for mouse NYU-45
3874
NYU-46
Successful for mouse NYU-46
3456
NYU-47
Successful for mouse NYU-47
5975
NYU-48
Successful for mouse NYU-48
2170
NYU-65
Successful for mouse NYU-65
6035
PL015
Successful for mouse PL015
34559
PL016
Successful for mouse PL016
21332
PL017
Successful for mouse PL017
3656
PL024
Successful for mouse PL024
3060
PL030
Successful for mouse PL030
3809
PL031
Successful for mouse PL031
8959
PL033
Successful for mouse PL033
18086